# FORGE-MA Combined: Red + Blue + TRL Self-Play

**Hackathon pipeline** — 4-API live consensus | regression guard | memory | leaderboard | GNN tracking

| Cell | Purpose |
|------|---------|
| 1 | API key setup (Colab secrets) |
| 2 | Install + repo clone |
| 3 | Load model (Qwen2.5-0.5B 4-bit) |
| 4 | Live SoT consensus (Groq + Cerebras + Mistral + OpenRouter) |
| 5 | Reward function (consensus + ground truth) |
| 6 | Dataset (200 evidence-enhanced episodes) |
| 7 | Memory + Regression Guard + Leaderboard + GNN |
| 8 | GRPO training with self-play loop |
| 9 | Evaluation + 4 plots (training, before/after, per-label, GNN) |


In [ ]:
# Cell 1 — API Keys from Colab Secrets
# Add these in Colab Secrets panel (left sidebar, key icon):
#   groqAPI         (Forensic Auditor)
#   CerebrasAPI     (Context Historian)
#   MistralAPI      (Narrative Critic)
#   OpenRouterAPI   (NegotiatedSearch)

import os
try:
    from google.colab import userdata
    _IN_COLAB = True
except ImportError:
    _IN_COLAB = False

# FIX: map Colab secret names → config.py expected env var names
SECRET_MAP = {
    'groqAPI':       ['OPENAI_API_KEY', 'OPENAI_API_KEY_AUDITOR'],
    'CerebrasAPI':   ['CEREBRAS_API_KEY', 'OPENAI_API_KEY_HISTORIAN'],
    'MistralAPI':    ['MISTRAL_API_KEY', 'OPENAI_API_KEY_CRITIC'],
    'OpenRouterAPI': ['OPENROUTER_API_KEY'],
}

loaded = []
if _IN_COLAB:
    for secret_name, env_vars in SECRET_MAP.items():
        try:
            val = userdata.get(secret_name)
            if val:
                for ev in env_vars:
                    os.environ[ev] = val
                loaded.append(secret_name)
                print(f'  ✅ {secret_name} → {", ".join(env_vars)}')
        except Exception as e:
            print(f'  ❌ {secret_name} not found ({e})')
else:
    print('Not in Colab — set env vars manually before running.')

print(f'\nLoaded {len(loaded)}/4 API keys: {loaded}')
if len(loaded) >= 3:
    print('✅ Sufficient for live SoT consensus')
else:
    print('⚠️  Will fall back to local-only rewards for missing providers')


In [ ]:
# Cell 2 — Install + Setup
import subprocess, sys, os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

subprocess.run(['pip', 'install', '-q',
    'unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git'],
    check=False)
subprocess.run(['pip', 'install', '-q',
    'trl', 'transformers', 'datasets', 'accelerate',
    'peft', 'torch-geometric', 'sentence-transformers',
    'openai', 'networkx'],
    check=False)

import torch, gc

WORK_DIR = '/content/forge-rl'
if not os.path.exists(WORK_DIR):
    os.system(f'git clone https://github.com/Godhand-Arnav/Scalar-finals.git {WORK_DIR}')
else:
    os.system(f'cd {WORK_DIR} && git pull origin main')

sys.modules['stix2'] = None
sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)

if not torch.cuda.is_available():
    raise RuntimeError('NO GPU. Runtime → Change runtime type → T4 GPU')

print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
print('Setup complete.')


In [ ]:
# Cell 3 — Load model
import gc, torch
from unsloth import FastLanguageModel

gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/Qwen2.5-0.5B-Instruct',
    max_seq_length=768,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16, lora_alpha=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    lora_dropout=0, bias='none',
    use_gradient_checkpointing='unsloth',
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

free, total = torch.cuda.mem_get_info(0)
print(f'Model loaded. VRAM: {free/1e9:.2f} GB free / {total/1e9:.2f} GB total')


In [ ]:
# Cell 4 — Live Society of Thought (4 LLM consensus)
# Calls Groq + Cerebras + Mistral + OpenRouter directly. No SoT class needed.
import re, json, threading, os
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FuturesTimeout

PROVIDERS = {
    'auditor': {
        'name':     'Forensic Auditor (Groq)',
        'env_key':  'OPENAI_API_KEY',
        'base_url': 'https://api.groq.com/openai/v1',
        'model':    'llama3-70b-8192',
        'system':   'You are a Forensic Auditor specialising in detecting fabricated sources, retracted studies, and misattributed quotes. Respond only in JSON with key: verdict (one of: real, misinfo, fabricated, satire, out_of_context).',
    },
    'historian': {
        'name':     'Context Historian (Cerebras)',
        'env_key':  'CEREBRAS_API_KEY',
        'base_url': 'https://api.cerebras.ai/v1',
        'model':    'llama3.1-70b',
        'system':   'You are a Context Historian specialising in detecting temporal manipulation, misdated media, and provenance fraud. Respond only in JSON with key: verdict (one of: real, misinfo, fabricated, satire, out_of_context).',
    },
    'critic': {
        'name':     'Narrative Critic (Mistral)',
        'env_key':  'MISTRAL_API_KEY',
        'base_url': 'https://api.mistral.ai/v1',
        'model':    'mistral-small-latest',
        'system':   'You are a Narrative Critic specialising in detecting satire, parody, and stylistic manipulation. Respond only in JSON with key: verdict (one of: real, misinfo, fabricated, satire, out_of_context).',
    },
    'searcher': {
        'name':     'Negotiated Search (OpenRouter)',
        'env_key':  'OPENROUTER_API_KEY',
        'base_url': 'https://openrouter.ai/api/v1',
        'model':    'meta-llama/llama-3-8b-instruct:free',
        'system':   'You are a search-based fact-checker specialising in cross-referencing claims. Respond only in JSON with key: verdict (one of: real, misinfo, fabricated, satire, out_of_context).',
    },
}

VERDICT_NORM = {
    'real': 'real', 'true': 'real', 'verified': 'real', 'legitimate': 'real',
    'misinfo': 'misinfo', 'misinformation': 'misinfo', 'false': 'misinfo',
    'satire': 'satire', 'parody': 'satire',
    'out_of_context': 'out_of_context', 'out of context': 'out_of_context',
    'fabricated': 'fabricated', 'fabrication': 'fabricated',
}

def _normalize(v):
    if not v: return 'unknown'
    return VERDICT_NORM.get(str(v).lower().strip().rstrip('.'), 'unknown')

# ── Build LLM clients once ────────────────────────────────────────────────────
from openai import OpenAI

_clients = {}
for role, cfg in PROVIDERS.items():
    api_key = os.environ.get(cfg['env_key'], '')
    if api_key and api_key != 'dummy-key':
        try:
            _clients[role] = {
                'client': OpenAI(base_url=cfg['base_url'], api_key=api_key),
                'cfg':    cfg,
                'live':   True,
            }
            print(f'  ✅ {cfg["name"]} ready')
        except Exception as e:
            print(f'  ❌ {cfg["name"]} init failed: {e}')
            _clients[role] = {'live': False}
    else:
        print(f'  ⚠️  {cfg["name"]} no key, will skip')
        _clients[role] = {'live': False}

def _call_one(role: str, claim: str, timeout: float = 8.0) -> str:
    """Call one LLM, return verdict string. Falls back to 'unknown' on any error."""
    c = _clients.get(role)
    if not c or not c.get('live'):
        return 'unknown'
    try:
        resp = c['client'].chat.completions.create(
            model=c['cfg']['model'],
            messages=[
                {'role': 'system', 'content': c['cfg']['system']},
                {'role': 'user',   'content': f'Classify this claim:\n{claim}\n\nReturn JSON: {{"verdict": "..."}}'},
            ],
            temperature=0.1,
            max_tokens=80,
            timeout=timeout,
        )
        text = resp.choices[0].message.content
        # Try JSON parse, fall back to regex
        try:
            obj = json.loads(re.search(r'\{.*\}', text, re.DOTALL).group())
            return _normalize(obj.get('verdict', 'unknown'))
        except Exception:
            for label in VERDICT_NORM.keys():
                if label in text.lower():
                    return _normalize(label)
            return 'unknown'
    except Exception:
        return 'unknown'

def sot_consensus(claim: str, timeout_per_agent: float = 8.0) -> dict:
    """Run 4 agents in parallel, return consensus dict."""
    verdicts = {}
    with ThreadPoolExecutor(max_workers=4) as ex:
        futures = {ex.submit(_call_one, role, claim, timeout_per_agent): role
                   for role in PROVIDERS.keys()}
        for fut in futures:
            role = futures[fut]
            try:
                verdicts[role] = fut.result(timeout=timeout_per_agent + 2)
            except Exception:
                verdicts[role] = 'unknown'

    # Count valid (non-unknown) votes
    votes = [v for v in verdicts.values() if v != 'unknown']
    if not votes:
        return {'consensus': 'unknown', 'level': 'no_signal', 'verdicts': verdicts, 'agree_count': 0}

    from collections import Counter
    counts = Counter(votes)
    top, top_count = counts.most_common(1)[0]
    n = len(votes)
    if top_count == n and n >= 3:
        level = 'unanimous'
    elif top_count >= 3:
        level = 'majority_3'
    elif top_count == 2 and n == 4:
        level = 'split_2_2'
    else:
        level = 'all_different'
    return {'consensus': top, 'level': level, 'verdicts': verdicts, 'agree_count': top_count}

# Quick test
print('\nTesting SoT on sample claim...')
_test = sot_consensus('Scientists at Harvard announced cure for cancer in 2026 press release.', timeout_per_agent=6)
print(f'  Consensus: {_test["consensus"]} ({_test["level"]}, {_test["agree_count"]}/4 agree)')
print(f'  Per-agent: {_test["verdicts"]}')


In [ ]:
# Cell 5 — Reward function (env truth + SoT consensus bonus)
import numpy as np, re, threading
from env.misinfo_env import MisInfoForensicsEnv

TOOL_ACTIONS = [0, 5, 2]  # query_source, temporal_audit, cross_reference

LABEL_TO_VERDICT = {
    'real': 8, 'verified': 8,
    'misinfo': 9, 'misinformation': 9,
    'satire': 10, 'parody': 10,
    'out_of_context': 11,
    'fabricated': 12,
}

VERDICT_TO_LABEL_OUT = {
    'MISINFO': 'misinfo', 'MISINFORMATION': 'misinfo',
    'SATIRE': 'satire',   'PARODY': 'satire',
    'VERIFIED': 'real',   'REAL': 'real',      'TRUE': 'real',
    'FABRICATED': 'fabricated', 'FABRICATION': 'fabricated',
    'OUT_OF_CONTEXT': 'out_of_context', 'OUT OF CONTEXT': 'out_of_context',
}

MISINFO_FAMILY = {'misinfo', 'satire', 'out_of_context', 'fabricated'}

def _parse_verdict_structured(text: str) -> str:
    m = re.search(
        r'VERDICT\s*:\s*(MISINFO|MISINFORMATION|SATIRE|PARODY|VERIFIED|REAL|TRUE|'
        r'FABRICATED|FABRICATION|OUT[\s_]OF[\s_]CONTEXT)',
        text.upper()
    )
    if m:
        raw = m.group(1).strip().replace(' ', '_')
        return VERDICT_TO_LABEL_OUT.get(raw, VERDICT_TO_LABEL_OUT.get(raw.replace('_', ' '), 'misinfo'))
    t = text.lower()
    if 'fabricat' in t:                                              return 'fabricated'
    if 'out of context' in t or 'recontextual' in t:                return 'out_of_context'
    if any(w in t for w in ['satire','parody','joke','humor']):     return 'satire'
    if any(w in t for w in ['this is verified','is verified',
                             'legitimate claim','true claim']):     return 'real'
    if any(w in t for w in ['misinfo','disinformation',
                             'mislead','manipulat']):                return 'misinfo'
    return 'misinfo'

def _label_to_action(label):
    return LABEL_TO_VERDICT.get(label.lower().strip(), 9)

def _compute_reward(predicted, true, sot_verdict=None, sot_level='no_signal'):
    """
    Hybrid reward: env truth + SoT consensus bonus.
    Base: 0.85 correct / 0.40 partial / 0.0 wrong
    Bonus: +0.10 if SoT agrees with prediction (validates reasoning)
    Penalty: -0.05 if SoT contradicts prediction
    """
    if predicted == true:
        base = 0.85
    elif predicted in MISINFO_FAMILY and true in MISINFO_FAMILY:
        base = 0.40
    else:
        base = 0.0

    bonus = 0.0
    if sot_verdict and sot_verdict != 'unknown':
        if sot_verdict == predicted:
            if sot_level == 'unanimous':   bonus = 0.10
            elif sot_level == 'majority_3': bonus = 0.07
            elif sot_level == 'split_2_2':  bonus = 0.03
        elif sot_verdict == true:  # SoT agreed with truth, model didn't
            bonus = -0.05
    return float(np.clip(base + bonus, 0.0, 1.0))

def _safe_step(env, action):
    try:
        result = env.step(action)
    except (AssertionError, Exception):
        return 0.0, True
    if isinstance(result, tuple):
        if len(result) == 5: return float(result[1]), bool(result[2] or result[3])
        if len(result) == 4: return float(result[1]), bool(result[2])
        if len(result) == 2: return float(result[1]), False
    return 0.0, True

def _safe_reset(env, seed=None):
    try:
        result = env.reset(seed=seed)
        return result if isinstance(result, tuple) and len(result) == 2 else (result, {})
    except Exception:
        return None, {}

def _make_env():
    return MisInfoForensicsEnv(difficulty=1)

def _get_claim_text(env):
    try:
        text, _, _ = env.get_graph_root_info()
        return text or ''
    except Exception:
        return ''

def _get_true_label(env):
    try:
        return env.get_graph_true_label()
    except Exception:
        return 'misinfo'

def _get_tool_evidence(env):
    evidence = {}
    tool_map = {0: 'query_source', 5: 'temporal_audit', 2: 'cross_reference'}
    for action, name in tool_map.items():
        try:
            result = env.tool_registry.call(name, env.graph)
            evidence[name] = result.get('summary', f'{name}: no result')
        except Exception:
            evidence[name] = f'{name}: unavailable'
    return evidence

# Persistent env + thread-safe seed counter
_seed_counter = [0]
_seed_lock = threading.Lock()
def _next_seed():
    with _seed_lock:
        _seed_counter[0] += 1
        return _seed_counter[0]

_REWARD_ENV = None
def _get_reward_env():
    global _REWARD_ENV
    if _REWARD_ENV is None:
        _REWARD_ENV = _make_env()
    return _REWARD_ENV

# ── Toggle: use live SoT during training? ─────────────────────────────────────
USE_SOT_IN_TRAINING = True   # False = faster, uses env truth only
SOT_TIMEOUT = 6.0

REWARD_DEBUG = True
_debug_step = [0]
_sot_calls = [0]

def reward_fn(prompts, completions, claim_texts=None, **kwargs):
    global _REWARD_ENV
    rewards = []
    env = _get_reward_env()

    for i, completion in enumerate(completions):
        comp_text = (completion[-1]['content']
                     if isinstance(completion, list) else str(completion))
        try:
            _, _ = _safe_reset(env, seed=_next_seed())
            done = False
            for act in TOOL_ACTIONS:
                if done: break
                _, done = _safe_step(env, act)
            true_label = _get_true_label(env)

            if not done:
                pred = _parse_verdict_structured(comp_text)
                sot_verdict, sot_level = None, 'no_signal'
                if USE_SOT_IN_TRAINING and i == 0:  # only call SoT once per group (cost)
                    claim_text = _get_claim_text(env) or 'Unknown claim'
                    try:
                        sot_result = sot_consensus(claim_text, timeout_per_agent=SOT_TIMEOUT)
                        sot_verdict = sot_result['consensus']
                        sot_level   = sot_result['level']
                        _sot_calls[0] += 1
                    except Exception:
                        pass
                reward = _compute_reward(pred, true_label, sot_verdict, sot_level)
                _safe_step(env, _label_to_action(pred))
            else:
                pred, reward = 'unknown', 0.0
            rewards.append(float(reward))

            if REWARD_DEBUG and _debug_step[0] < 30:
                _debug_step[0] += 1
                icon = '✅' if reward >= 0.8 else '⚠️' if reward >= 0.4 else '❌'
                preview = comp_text[:55].replace('\n', ' ')
                print(f'[dbg #{_debug_step[0]:02d}] {icon} '
                      f'true={true_label} pred={pred} r={reward:.3f} | "{preview}..."')
        except Exception as e:
            print(f'[reward_fn] ep {i} error: {e}')
            try: _REWARD_ENV = _make_env()
            except: pass
            rewards.append(0.0)
    return rewards

# Sanity check
print('Sanity check (5 seeds)...')
_passed = 0
_test_env = _make_env()
for _s in range(5):
    try:
        _, _ = _safe_reset(_test_env, seed=_s)
        _true = _get_true_label(_test_env)
        _done = False
        for _a in TOOL_ACTIONS:
            if _done: break
            _, _done = _safe_step(_test_env, _a)
        _r = _compute_reward(_true, _true) if not _done else 0.0
        if not _done: _safe_step(_test_env, _label_to_action(_true))
        print(f'  {"✅" if _r >= 0.8 else "❌"} seed={_s} label={_true} r={_r:.3f}')
        if _r >= 0.8: _passed += 1
    except Exception as _e:
        print(f'  ❌ seed={_s} error: {_e}')
print(f'\n{"✅ PASSED" if _passed >= 3 else "❌ FAILED"} ({_passed}/5)')


In [ ]:
# Cell 6 — Dataset with investigation evidence
from datasets import Dataset
import random

PROMPT_TMPL = """You are a misinformation forensics agent.

CLAIM: {claim}

INVESTIGATION FINDINGS:
  Source analysis:  {source_summary}
  Temporal audit:   {temporal_summary}
  Cross-reference:  {crossref_summary}

Based on the investigation findings, classify this claim.
Respond in this exact format:
VERDICT: [MISINFO|SATIRE|VERIFIED|FABRICATED|OUT_OF_CONTEXT]
REASON: [one sentence explaining your verdict]"""

TASK_NAMES = [
    'fabricated_stats', 'out_of_context', 'coordinated_campaign',
    'satire_news', 'verified_fact', 'politifact_liar',
    'image_forensics', 'sec_fraud'
]

print('Building dataset (200 episodes)...')
random.seed(42)
claims, prompts, true_labels_list = [], [], []
label_counts = {}

_ds_env = _make_env()
for i in range(200):
    try:
        _, _ = _safe_reset(_ds_env, seed=i)
        claim = _get_claim_text(_ds_env) or f'Claim #{i}: {TASK_NAMES[i % len(TASK_NAMES)]}'
        true_label = _get_true_label(_ds_env)
        label_counts[true_label] = label_counts.get(true_label, 0) + 1
        evidence = _get_tool_evidence(_ds_env)
        prompt_text = PROMPT_TMPL.format(
            claim=claim,
            source_summary=evidence.get('query_source',   'Source data unavailable'),
            temporal_summary=evidence.get('temporal_audit', 'Temporal data unavailable'),
            crossref_summary=evidence.get('cross_reference','Cross-reference unavailable'),
        )
    except Exception as e:
        if i < 3: print(f'  ep {i} error: {e}')
        claim = f'Claim #{i}: {TASK_NAMES[i % len(TASK_NAMES)]}'
        true_label = 'misinfo'
        label_counts['misinfo'] = label_counts.get('misinfo', 0) + 1
        prompt_text = PROMPT_TMPL.format(
            claim=claim,
            source_summary='Source has low trust score (0.18).',
            temporal_summary='Temporal anomaly detected.',
            crossref_summary='Cross-reference revealed 2 contradicting sources.',
        )
    claims.append(claim)
    true_labels_list.append(true_label)
    prompts.append([{'role': 'user', 'content': prompt_text}])
    if (i + 1) % 50 == 0:
        print(f'  {i+1}/200 | labels={label_counts}')

dataset = Dataset.from_dict({
    'prompt': prompts, 'claim_texts': claims, 'true_labels': true_labels_list
})
print(f'\nDataset: {len(dataset)} rows | labels={label_counts}')


In [ ]:
# Cell 7 — Memory + Regression Guard + Leaderboard + GNN
# Implements 4 of the 5 hackathon upgrade modules
import networkx as nx
from collections import defaultdict

# ── 1. MEMORY SYSTEM ──────────────────────────────────────────────────────────
class Memory:
    def __init__(self):
        self.best_score = -float('inf')
        self.best_checkpoint_path = None
        self.best_iteration = 0
        self.history = []   # list of dicts: {iter, score, accepted}
        self.failed_strategies = set()  # patterns that consistently underperformed

    def update(self, iteration, score, checkpoint_path=None, tolerance=0.01):
        """Regression Guard: only accept if score >= best - tolerance."""
        accepted = score >= (self.best_score - tolerance)
        entry = {
            'iteration':  iteration,
            'score':      score,
            'accepted':   accepted,
            'best_score': self.best_score,
        }
        if accepted and score > self.best_score:
            self.best_score = score
            self.best_checkpoint_path = checkpoint_path
            self.best_iteration = iteration
            entry['promoted'] = True
        else:
            entry['promoted'] = False
        self.history.append(entry)
        return accepted

    def summary(self):
        n = len(self.history)
        accepted = sum(1 for h in self.history if h['accepted'])
        promoted = sum(1 for h in self.history if h.get('promoted'))
        return {
            'iterations':   n,
            'accepted':     accepted,
            'promoted':     promoted,
            'best_score':   self.best_score,
            'best_iter':    self.best_iteration,
            'reject_rate':  (n - accepted) / max(1, n),
        }

memory = Memory()

# ── 2. LEADERBOARD ────────────────────────────────────────────────────────────
class Leaderboard:
    def __init__(self):
        self.entries = []   # list of (iter, score, label, accepted)

    def add(self, iteration, score, label='checkpoint', accepted=True):
        self.entries.append({
            'iteration': iteration, 'score': score,
            'label': label, 'accepted': accepted,
        })

    def top_n(self, n=5):
        return sorted(self.entries, key=lambda e: -e['score'])[:n]

    def display(self, top_n=5):
        print(f'\n{"="*55}')
        print(f'LEADERBOARD (top {top_n})')
        print(f'{"="*55}')
        for rank, e in enumerate(self.top_n(top_n), 1):
            mark = '✅' if e['accepted'] else '❌'
            print(f'  #{rank}  iter={e["iteration"]:3d}  score={e["score"]:.4f}  '
                  f'{mark} {e["label"]}')
        print(f'{"="*55}\n')

leaderboard = Leaderboard()

# ── 3. INTERACTION GRAPH (GNN-ready) ──────────────────────────────────────────
class InteractionGraph:
    """
    Track Red→Blue interactions as a graph.
    Nodes: tasks, red_outputs, blue_outputs, scores
    Edges: attack, defense, scored_as
    GNN-ready: each node has features dict
    """
    def __init__(self):
        self.G = nx.DiGraph()
        self.task_count = 0
        self.cluster_failures = defaultdict(int)  # task_type → fail count

    def add_interaction(self, task_id, task_type, red_attack, blue_response,
                        score, true_label, predicted):
        """Add one Red→Blue→score interaction to the graph."""
        task_node = f't_{task_id}'
        red_node  = f'r_{task_id}'
        blue_node = f'b_{task_id}'

        self.G.add_node(task_node,
                        type='task', task_type=task_type,
                        true_label=true_label,
                        score=score)
        self.G.add_node(red_node,
                        type='red_output',
                        attack_summary=str(red_attack)[:60],
                        embedding=None)  # GNN can fill later
        self.G.add_node(blue_node,
                        type='blue_output',
                        predicted=predicted,
                        score=score,
                        correct=(predicted == true_label))

        self.G.add_edge(task_node, red_node, type='attack')
        self.G.add_edge(red_node, blue_node, type='defense')
        self.G.add_edge(blue_node, task_node, type='scored', score=score)

        if predicted != true_label:
            self.cluster_failures[task_type] += 1
        self.task_count += 1

    def failure_clusters(self, top_n=3):
        """Return task types with highest failure counts."""
        return sorted(self.cluster_failures.items(),
                      key=lambda x: -x[1])[:top_n]

    def stats(self):
        return {
            'nodes':           self.G.number_of_nodes(),
            'edges':           self.G.number_of_edges(),
            'tasks':           self.task_count,
            'failure_types':   len(self.cluster_failures),
            'top_failures':    self.failure_clusters(3),
        }

interaction_graph = InteractionGraph()

print('Memory + Regression Guard + Leaderboard + GNN initialized.')
print('  - Memory: tracks best_score, rejects regressions')
print('  - Leaderboard: ranks all iterations')
print('  - InteractionGraph: stores Red→Blue→score as DiGraph (GNN-ready)')


In [ ]:
# Cell 8 — Baseline → Self-Play GRPO Training
import gc, torch, inspect, warnings, numpy as np
from trl import GRPOConfig, GRPOTrainer

warnings.filterwarnings('ignore', category=FutureWarning, module='transformers')

# ── BASELINE FIRST (anchor for memory) ────────────────────────────────────────
def evaluate_heuristic(n_episodes=20):
    rewards, correct = [], 0
    env = _make_env()
    for i in range(n_episodes):
        try:
            _, _ = _safe_reset(env, seed=100 + i)
            true_label = _get_true_label(env)
            done = False
            for act in TOOL_ACTIONS:
                if done: break
                _, done = _safe_step(env, act)
            r = _compute_reward('misinfo', true_label) if not done else 0.0
            if not _safe_step(env, 9)[1]: pass
            rewards.append(r)
            if r >= 0.8: correct += 1
        except Exception:
            rewards.append(0.0)
    return float(np.mean(rewards)), correct / n_episodes

baseline_reward, baseline_acc = evaluate_heuristic(20)
print(f'BASELINE — reward: {baseline_reward:.4f} | acc: {baseline_acc:.2%}')

# Anchor in memory
memory.update(iteration=0, score=baseline_reward, checkpoint_path=None)
leaderboard.add(0, baseline_reward, label='heuristic_baseline', accepted=True)

# ── GRPO CONFIG ───────────────────────────────────────────────────────────────
gc.collect()
torch.cuda.empty_cache()

_gpu = torch.cuda.get_device_name(0)
_use_bf16 = any(x in _gpu for x in ['A100','A10','H100','RTX 30','RTX 40'])
_use_fp16 = not _use_bf16

_valid = set(inspect.signature(GRPOConfig).parameters.keys())
_cfg = dict(
    output_dir='/content/forge-grpo',
    max_steps=300,
    num_generations=4,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=3e-6,
    save_steps=100,
    logging_steps=5,
    report_to='none',
    warmup_ratio=0.15,
    bf16=_use_bf16,
    fp16=_use_fp16,
    dataloader_pin_memory=False,
)
for k, v in [('max_completion_length', 80), ('max_new_tokens', 80)]:
    if k in _valid: _cfg[k] = v; break
if 'max_prompt_length' in _valid: _cfg['max_prompt_length'] = 512
for k in ('kl_coef', 'beta'):
    if k in _valid: _cfg[k] = 0.01; break
if 'temperature' in _valid: _cfg['temperature'] = 0.9

config = GRPOConfig(**_cfg)
print(f'\nsteps={config.max_steps} | generations={config.num_generations} | '
      f'eff_batch={config.per_device_train_batch_size * config.gradient_accumulation_steps}')

# ── SELF-PLAY CALLBACK: log every checkpoint to memory + leaderboard ─────────
from transformers import TrainerCallback

class SelfPlayCallback(TrainerCallback):
    """Self-Play: each checkpoint is a new iteration. Memory+regression guard apply."""
    def __init__(self):
        self.last_score = baseline_reward
        self.iter = 0

    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs: return
        # Pull current mean reward from logs
        cur_reward = None
        for key in ('rewards/mean', 'reward/mean', 'mean_reward'):
            if key in logs:
                cur_reward = logs[key]
                break
        if cur_reward is None: return

        self.iter += 1
        accepted = memory.update(self.iter, cur_reward, tolerance=0.01)
        leaderboard.add(self.iter, cur_reward, label=f'step_{state.global_step}', accepted=accepted)

        if not accepted:
            # Regression guard fired — log but don't actually rollback (LoRA in flight)
            print(f'  [GUARD] step={state.global_step} score={cur_reward:.4f} '
                  f'< best={memory.best_score:.4f} — flagged as regression')

# ── TRAIN ─────────────────────────────────────────────────────────────────────
try:
    trainer = GRPOTrainer(
        model=model, reward_funcs=reward_fn, args=config,
        train_dataset=dataset, processing_class=tokenizer,
        callbacks=[SelfPlayCallback()],
    )
except TypeError:
    trainer = GRPOTrainer(
        model=model, reward_funcs=reward_fn, args=config,
        train_dataset=dataset, tokenizer=tokenizer,
        callbacks=[SelfPlayCallback()],
    )

print('\nStarting self-play GRPO training...')
print(f'SoT enabled: {USE_SOT_IN_TRAINING} | LLM calls so far: {_sot_calls[0]}\n')

try:
    trainer.train()
    print('\nTraining complete.')
    trainer.save_model('/content/forge-grpo-final')
    tokenizer.save_pretrained('/content/forge-grpo-final')
except KeyboardInterrupt:
    print('\nInterrupted — saving...')
    trainer.save_model('/content/forge-grpo-interrupted')
except Exception as e:
    print(f'\nError: {e}')
    raise

print(f'\nTotal SoT calls during training: {_sot_calls[0]}')
print(f'Memory summary: {memory.summary()}')
leaderboard.display(top_n=5)


In [ ]:
# Cell 9 — Evaluation + 4 Plots
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np, torch, os
from unsloth import FastLanguageModel

os.makedirs('/content/results', exist_ok=True)
FastLanguageModel.for_inference(model)
print('Model in inference mode.')

# ── Final Evaluation (with SoT consensus tracking + GNN logging) ─────────────

def evaluate_trained(n_episodes=30, log_to_gnn=True):
    rewards, correct = [], 0
    env = _make_env()
    label_results, episode_log = {}, []
    sot_agreement = {'agree_correct': 0, 'agree_wrong': 0, 'disagree': 0, 'no_signal': 0}
    print(f'Evaluating {n_episodes} episodes (with live SoT)...')

    for i in range(n_episodes):
        try:
            _, _ = _safe_reset(env, seed=200 + i)
            claim = _get_claim_text(env) or f'Claim #{200+i}'
            true_label = _get_true_label(env)
            evidence = _get_tool_evidence(env)

            prompt_text = PROMPT_TMPL.format(
                claim=claim,
                source_summary=evidence.get('query_source',   'Unavailable'),
                temporal_summary=evidence.get('temporal_audit', 'Unavailable'),
                crossref_summary=evidence.get('cross_reference','Unavailable'),
            )
            prompt_ids = tokenizer.apply_chat_template(
                [{'role': 'user', 'content': prompt_text}],
                tokenize=True, return_tensors='pt', add_generation_prompt=True,
            ).to('cuda')
            with torch.no_grad():
                outputs = model.generate(
                    input_ids=prompt_ids, max_new_tokens=80,
                    temperature=0.1, do_sample=True,
                )
            response = tokenizer.decode(outputs[0][prompt_ids.shape[1]:], skip_special_tokens=True)
            del prompt_ids, outputs

            done = False
            for act in TOOL_ACTIONS:
                if done: break
                _, done = _safe_step(env, act)
            pred = _parse_verdict_structured(response)

            # Live SoT for evaluation
            sot = sot_consensus(claim, timeout_per_agent=6.0)
            sot_v, sot_l = sot['consensus'], sot['level']
            reward = _compute_reward(pred, true_label, sot_v, sot_l)
            if not done: _safe_step(env, _label_to_action(pred))

            # SoT agreement tracking
            if sot_v == 'unknown':
                sot_agreement['no_signal'] += 1
            elif sot_v == pred and pred == true_label:
                sot_agreement['agree_correct'] += 1
            elif sot_v == pred and pred != true_label:
                sot_agreement['agree_wrong'] += 1
            else:
                sot_agreement['disagree'] += 1

            rewards.append(reward)
            if reward >= 0.8: correct += 1
            label_results.setdefault(true_label, {'correct': 0, 'total': 0})
            label_results[true_label]['total'] += 1
            if reward >= 0.8: label_results[true_label]['correct'] += 1
            episode_log.append({
                'ep': i, 'true': true_label, 'pred': pred, 'reward': reward,
                'sot': sot_v, 'sot_level': sot_l
            })

            # GNN logging
            if log_to_gnn:
                interaction_graph.add_interaction(
                    task_id=i,
                    task_type=TASK_NAMES[i % len(TASK_NAMES)],
                    red_attack=claim[:60],
                    blue_response=pred,
                    score=reward,
                    true_label=true_label,
                    predicted=pred,
                )

            if (i + 1) % 5 == 0:
                icon = '✅' if reward >= 0.8 else '⚠️' if reward >= 0.4 else '❌'
                print(f'  ep {i+1}/{n_episodes} {icon} | true={true_label:12s} '
                      f'pred={pred:12s} sot={sot_v:12s} r={reward:.3f} | '
                      f'acc={correct/(i+1):.1%}')
        except Exception as e:
            print(f'  eval error ep {i}: {e}')
            rewards.append(0.0)

    mean_r = float(np.mean(rewards))
    acc = correct / n_episodes
    print(f'\nPer-label accuracy:')
    for label, res in sorted(label_results.items()):
        pct = res['correct'] / res['total'] if res['total'] > 0 else 0
        bar = '█' * int(pct * 20)
        print(f'  {label:20s}: {res["correct"]}/{res["total"]:2d} ({pct:.0%}) {bar}')
    print(f'\nSoT agreement: {sot_agreement}')
    return mean_r, acc, episode_log, label_results, sot_agreement

trained_reward, trained_acc, episode_log, label_results, sot_agree = \
    evaluate_trained(n_episodes=30)

target_met = trained_acc >= 0.80
print(f'\n{"="*55}')
print(f'BASELINE — reward: {baseline_reward:.4f} | acc: {baseline_acc:.2%}')
print(f'TRAINED  — reward: {trained_reward:.4f} | acc: {trained_acc:.2%}')
print(f'DELTA    — {trained_reward - baseline_reward:+.4f} | '
      f'{trained_acc - baseline_acc:+.2%}')
print(f'TARGET:  {"✅ 80% MET" if target_met else f"❌ {0.80 - trained_acc:.0%} below"}')
print(f'{"="*55}')
print(f'\nGNN graph stats: {interaction_graph.stats()}')

# ── Extract training reward curve ────────────────────────────────────────────
steps, rew_log, losses, loss_steps = [], [], [], []
for l in trainer.state.log_history:
    if 'step' not in l: continue
    for key in ('rewards/mean', 'reward/mean', 'train/rewards/mean', 'mean_reward'):
        if key in l:
            steps.append(l['step']); rew_log.append(l[key]); break
    if 'loss' in l:
        loss_steps.append(l['step']); losses.append(l['loss'])

# ── Plot 1: Training curve + loss ─────────────────────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
ax = axes[0]
if steps:
    ax.plot(steps, rew_log, color='#3498db', linewidth=2.5, label='Reward')
    if len(rew_log) >= 5:
        w = max(3, len(rew_log) // 8)
        rolled = np.convolve(rew_log, np.ones(w)/w, mode='valid')
        ax.plot(steps[w-1:], rolled, '#1a5276', linewidth=1.5, linestyle='--',
                alpha=0.8, label=f'Rolling avg (w={w})')
ax.axhline(y=baseline_reward, color='#e74c3c', linestyle='--', linewidth=2,
           label=f'Baseline ({baseline_reward:.3f})')
ax.axhline(y=0.80, color='#27ae60', linestyle=':', linewidth=2, label='80% target')
ax.set_ylabel('Mean Reward', fontsize=12)
ax.set_title('FORGE-MA Combined: Self-Play GRPO Training', fontsize=13, fontweight='bold')
ax.legend(fontsize=10); ax.grid(True, alpha=0.3)
ax.set_ylim(min(0, min(rew_log)-0.05) if rew_log else 0, 1.05)

ax2 = axes[1]
if losses:
    ax2.plot(loss_steps, losses, color='#e67e22', linewidth=2, label='Training loss')
    ax2.set_ylabel('Loss', fontsize=12)
    ax2.legend(fontsize=10); ax2.grid(True, alpha=0.3)
    ax2.set_title(f'Loss (min={min(losses):.4f})', fontsize=11)
ax2.set_xlabel('Training Step', fontsize=12)
plt.tight_layout()
plt.savefig('/content/results/training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curve.png')

# ── Plot 2: Before/After + Per-label ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
colors = ['#e74c3c', '#2ecc71' if target_met else '#f39c12']
bars = ax.bar(['Heuristic\nBaseline', 'GRPO\nTrained'],
              [baseline_reward, trained_reward],
              color=colors, edgecolor='black', linewidth=1.2, width=0.5)
ax.axhline(y=0.80, color='#27ae60', linestyle=':', linewidth=2.5, label='80% target')
ax.set_ylabel('Mean Reward', fontsize=12)
ax.set_title('Before vs After Self-Play GRPO', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.1); ax.legend(fontsize=11); ax.grid(True, alpha=0.2, axis='y')
for bar, val in zip(bars, [baseline_reward, trained_reward]):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.025,
            f'{val:.3f}', ha='center', va='bottom', fontsize=15, fontweight='bold')

ax2 = axes[1]
if label_results:
    ll = sorted(label_results.keys())
    accs = [label_results[l]['correct']/label_results[l]['total']
            if label_results[l]['total'] > 0 else 0 for l in ll]
    ns = [label_results[l]['total'] for l in ll]
    cols = ['#2ecc71' if a >= 0.8 else '#f39c12' if a >= 0.5 else '#e74c3c' for a in accs]
    bars2 = ax2.barh(ll, accs, color=cols, edgecolor='black', linewidth=0.8)
    ax2.axvline(x=0.80, color='#27ae60', linestyle=':', linewidth=2.5)
    ax2.set_xlabel('Accuracy', fontsize=12)
    ax2.set_title('Per-Label Accuracy', fontsize=13, fontweight='bold')
    ax2.set_xlim(0, 1.2); ax2.grid(True, alpha=0.2, axis='x')
    for bar, acc, n in zip(bars2, accs, ns):
        ax2.text(acc + 0.02, bar.get_y() + bar.get_height()/2,
                 f'{acc:.0%} (n={n})', va='center', fontsize=10, fontweight='bold')
plt.suptitle(f'FORGE-MA | Acc: {trained_acc:.1%} | {"✅ TARGET MET" if target_met else "❌ NOT YET"}',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/content/results/before_after.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: before_after.png')

# ── Plot 3: SoT consensus pie + leaderboard ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
sot_labels  = ['Agree+Correct', 'Agree+Wrong', 'Disagree', 'No SoT signal']
sot_values  = [sot_agree['agree_correct'], sot_agree['agree_wrong'],
               sot_agree['disagree'], sot_agree['no_signal']]
sot_colors  = ['#2ecc71', '#e67e22', '#3498db', '#95a5a6']
ax.pie([v for v in sot_values if v > 0],
       labels=[l for l, v in zip(sot_labels, sot_values) if v > 0],
       colors=[c for c, v in zip(sot_colors, sot_values) if v > 0],
       autopct='%1.0f%%', startangle=90)
ax.set_title('SoT 4-Agent Consensus vs Model Predictions', fontsize=13, fontweight='bold')

ax2 = axes[1]
top5 = leaderboard.top_n(8)
y_pos = range(len(top5))
scores = [e['score'] for e in top5]
labels_lb = [f"#{r}: iter {e['iteration']}" for r, e in enumerate(top5, 1)]
cols = ['#2ecc71' if e['score'] >= 0.8 else '#f39c12' if e['score'] >= 0.5 else '#e74c3c' for e in top5]
ax2.barh(y_pos, scores, color=cols, edgecolor='black')
ax2.set_yticks(y_pos); ax2.set_yticklabels(labels_lb, fontsize=10)
ax2.invert_yaxis()
ax2.set_xlabel('Score', fontsize=12)
ax2.set_title('Top 8 Leaderboard', fontsize=13, fontweight='bold')
ax2.axvline(x=0.80, color='#27ae60', linestyle=':', linewidth=2)
ax2.grid(True, alpha=0.2, axis='x')
for i, s in enumerate(scores):
    ax2.text(s + 0.005, i, f'{s:.3f}', va='center', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('/content/results/sot_leaderboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: sot_leaderboard.png')

# ── Plot 4: GNN interaction graph ─────────────────────────────────────────────
import networkx as nx
fig, ax = plt.subplots(figsize=(11, 7))
G = interaction_graph.G
# Sample subgraph (full graph too dense for plot)
sample_tasks = [n for n in G.nodes() if n.startswith('t_')][:15]
sub_nodes = set(sample_tasks)
for t in sample_tasks:
    sub_nodes.update(G.successors(t))
    for s in G.successors(t):
        sub_nodes.update(G.successors(s))
SG = G.subgraph(sub_nodes)

pos = nx.spring_layout(SG, seed=42, k=1.5)
node_colors = []
for n in SG.nodes():
    nt = SG.nodes[n].get('type', '')
    if nt == 'task':       node_colors.append('#3498db')
    elif nt == 'red_output':  node_colors.append('#e74c3c')
    elif nt == 'blue_output':
        correct = SG.nodes[n].get('correct', False)
        node_colors.append('#2ecc71' if correct else '#f39c12')
    else: node_colors.append('#95a5a6')

nx.draw_networkx_edges(SG, pos, ax=ax, alpha=0.4, arrows=True, edge_color='#7f8c8d')
nx.draw_networkx_nodes(SG, pos, ax=ax, node_color=node_colors,
                       node_size=400, edgecolors='black', linewidths=0.8)

# Legend
legend_patches = [
    mpatches.Patch(color='#3498db', label='Task'),
    mpatches.Patch(color='#e74c3c', label='Red attack'),
    mpatches.Patch(color='#2ecc71', label='Blue (correct)'),
    mpatches.Patch(color='#f39c12', label='Blue (wrong)'),
]
ax.legend(handles=legend_patches, loc='upper left', fontsize=10)
stats = interaction_graph.stats()
ax.set_title(f'FORGE-MA Interaction Graph (sample of {len(sample_tasks)} tasks)\n'
             f'Total: {stats["nodes"]} nodes, {stats["edges"]} edges, '
             f'{stats["failure_types"]} failure clusters',
             fontsize=12, fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('/content/results/interaction_graph.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: interaction_graph.png')

print('\nAll 4 plots saved. Self-play complete.')
print(f'Top failure clusters: {interaction_graph.failure_clusters(3)}')
